# 0.1 Import Libraries

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from bioimage_pipeline.data import (
    find_tiff_files,
    get_image_statistics,
    list_ctc_sequences,
    load_tiff_image,
    summarize_dataset_structure,
)
from bioimage_pipeline.visualization import (
    plot_max_intensity_projection,
    plot_slice,
    save_figure,
)

# 0.2 Set Paths

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

MDA231_DIR = PROJECT_ROOT / "data" / "raw" / "Fluo-C3DL-MDA231"
A549_DIR = PROJECT_ROOT / "data" / "raw" / "Fluo-C3DH-A549"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"

# 1.1 Inspect MDA231 Dataset Structure

In [ ]:
mda231_summary = summarize_dataset_structure(MDA231_DIR)
mda231_summary

# 1.2 Inspect A549 Dataset Structure

In [ ]:
a549_summary = summarize_dataset_structure(A549_DIR)
a549_summary

# 2.1 Load Sample TIFF Image

In [ ]:
sequences = list_ctc_sequences(MDA231_DIR)
if not sequences:
    raise FileNotFoundError(f"No sequence folders found in {MDA231_DIR}")

sample_files = find_tiff_files(sequences[0])
if not sample_files:
    raise FileNotFoundError(f"No TIFF files found in {sequences[0]}")

sample_path = sample_files[0]
sample_image = load_tiff_image(sample_path)
sample_path

# 2.2 Check Image Shape and Intensity Range

In [ ]:
image_statistics = get_image_statistics(sample_image)
image_statistics

# 2.3 Visualize One Z-Slice

In [ ]:
slice_figure = None
if sample_image.ndim == 2:
    display_image = sample_image
    slice_title = f"2D Image: {sample_path.name}"
elif sample_image.ndim == 3:
    middle_z = sample_image.shape[0] // 2
    display_image = sample_image[middle_z]
    slice_title = f"Middle Z-Slice ({middle_z}): {sample_path.name}"
else:
    print(f"Warning: unexpected image shape {sample_image.shape}")

if sample_image.ndim in (2, 3):
    slice_figure, slice_axis = plt.subplots(figsize=(7, 7))
    plot_slice(display_image, ax=slice_axis, title=slice_title)
    plt.show()

# 2.4 Visualize Max Intensity Projection

In [ ]:
projection_figure = None
if sample_image.ndim in (2, 3):
    projection_figure, projection_axis = plt.subplots(figsize=(7, 7))
    plot_max_intensity_projection(
        sample_image,
        ax=projection_axis,
        title=f"Max Intensity Projection: {sample_path.name}",
    )
    plt.show()
else:
    print(f"Warning: cannot project image with shape {sample_image.shape}")

# 3.1 Check Ground Truth Availability

In [ ]:
mda231_summary[
    mda231_summary["folder_type"].isin(["GT", "SEG", "TRA"])
][["sequence", "folder_type", "exists", "tiff_count", "folder_path"]]

# 4.1 Save Sample Figures

In [ ]:
saved_figures = []
if slice_figure is not None:
    saved_figures.append(save_figure(slice_figure, "mda231_sample_z_slice.png", FIGURES_DIR))
if projection_figure is not None:
    saved_figures.append(
        save_figure(projection_figure, "mda231_sample_max_projection.png", FIGURES_DIR)
    )
saved_figures

# 5.1 Summary and Next Steps

Record observed image dimensions, intensity characteristics, and annotation availability before defining preprocessing or segmentation choices.